In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import seaborn as sns

In [2]:
from vaccination_functions import *
from tea_data_functions import *

In [3]:
tx_mmr_vac_rates = pd.read_csv("../TX_MMR_vax_rate_24_25.csv")
tx_mmr_vac_rates.head()
#counties_with_vac = tx_mmr_vac_rates['County'].unique()

,Facility Number,School Type,School District or Name,Facility Address,County,MMR Vaccination Rate,Age Group
0,9057829000,Public School,A+ ACADEMY,"8225 BRUTON RD, DALLAS, TX, .",Dallas,96.77,Kindergarten
1,9109901000,Public School,ABBOTT ISD,"P O BOX 226, ABBOTT, TX, .",Hill,95.24,Kindergarten
2,9095901000,Public School,ABERNATHY ISD,"505 7TH ST, ABERNATHY, TX, .",Hale,93.22,Kindergarten
3,9221901000,Public School,ABILENE ISD,"P O BOX 981, ABILENE, TX, .",Taylor,97.76,Kindergarten
4,9014901000,Public School,ACADEMY ISD,"704 E MAIN ST, LITTLE RIVER, TX, .",Bell,98.04,Kindergarten


In [4]:
#disticts_with_vac = tx_mmr_vac_rates['School District or Name'].unique()
#disticts_with_vac

In [5]:
tea_data_path = "../TEA_data/"
econ_disadv = pd.read_csv(tea_data_path+"Economically_Disadvantaged_Report_Statewide_Districts_2024_2025.csv")
grade_enrollment = pd.read_csv(tea_data_path+"Enrollment_Report_Statewide_Districts_Grade_2024_2025.csv")
gender_enrollment = pd.read_csv(tea_data_path+"Enrollment_Report_Statewide_Districts_Gender_2024_2025.csv")
ethnicity_enrollment = pd.read_csv(tea_data_path+"Enrollment_Report_Statewide_Districts_Ethnicity_2024_2025.csv")
staff_district = pd.read_csv(tea_data_path+"Staff_Salary_FTE_Report_Statewide_Districts_2024_2025.csv")
#district_type.head()

In [6]:
econ_disadv.head()

,Region,County Name,District Name,District Number,Charter Status,Eligible For Free Meals Count,Eligible For Free Meals Percent,Eligible For Reduced-Price Meals Count,Eligible For Reduced-Price Meals Percent,Other Economically Disadvantaged Count,Other Economically Disadvantaged Percent,Not Economically Disadvantaged Count,Not Economically Disadvantaged Percent,Total Count
0,10,DALLAS COUNTY,A+ ACADEMY,57829.0,OPEN ENROLLMENT CHARTER,1440.0,85.82,150.0,8.94,0.0,0.00,88.0,5.24,1678.0
1,4,HARRIS COUNTY,A+ UNLIMITED POTENTIAL,101871.0,OPEN ENROLLMENT CHARTER,141.0,69.46,20.0,9.85,0.0,0.00,42.0,20.69,203.0
2,12,HILL COUNTY,ABBOTT ISD,109901.0,TRADITIONAL ISD/CSD,87.0,30.63,22.0,7.75,0.0,0.00,175.0,61.62,284.0
3,17,HALE COUNTY,ABERNATHY ISD,95901.0,TRADITIONAL ISD/CSD,297.0,36.76,-999.0,-999.00,-999.0,-999.00,415.0,51.36,808.0
4,14,TAYLOR COUNTY,ABILENE ISD,221901.0,TRADITIONAL ISD/CSD,7205.0,49.57,134.0,0.92,2652.0,18.24,4545.0,31.27,14536.0


In [7]:
staff_district.head()

,Region,County,District,District Name,Charter Status,Staff_Category,Staff,FTE Count,Total Base Pay,Average Base Pay
0,10,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,TEACHING STAFF,KINDERGARTEN,4.04,278424.0,68946
1,10,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,TEACHING STAFF,KINDERGARTEN/ELEMENTARY (K-6),8.18,577027.0,70541
2,10,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,TEACHING STAFF,MIDDLE SCHOOL (GRADES 6 - 8),2.91,218878.0,75203
3,10,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,TEACHING STAFF,ALL GRADE LEVELS,5.64,421021.0,74686
4,10,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,TEACHING STAFF,GRADE 1,4.99,365122.0,73130


In [8]:
staff_district.columns

Index(['Region', 'County', 'District', 'District Name', ' Charter Status',
       'Staff_Category', 'Staff', 'FTE Count', 'Total Base Pay',
       'Average Base Pay'],
      dtype='str')

In [9]:
staff_column_drop = ['Region', 'District', ' Charter Status', 'Staff_Category', 'Staff']
staff_district_clean = staff_district.drop(columns=staff_column_drop).dropna().rename(columns={'County': 'County_Name', 'District Name': 'District_Name'})
staff_district_clean.head()

,County_Name,District_Name,FTE Count,Total Base Pay,Average Base Pay
0,DALLAS COUNTY,A+ ACADEMY,4.04,278424.0,68946
1,DALLAS COUNTY,A+ ACADEMY,8.18,577027.0,70541
2,DALLAS COUNTY,A+ ACADEMY,2.91,218878.0,75203
3,DALLAS COUNTY,A+ ACADEMY,5.64,421021.0,74686
4,DALLAS COUNTY,A+ ACADEMY,4.99,365122.0,73130


In [10]:
econ_rename_dict = {' County Name': 'County_Name', ' District Name': 'District_Name', ' Total Count': 'Total_Count'}
econ_disadv_clean = econ_disadv.drop(columns=['Region', ' District Number', ' Charter Status']).rename(columns=econ_rename_dict).dropna()
common_column_drop = ["CHARTER STATUS", 'REGION', 'YEAR', 'DISTRICT']
enrollment_index_col = ['COUNTY NAME', 'DISTRICT NAME']
grade_enrollment_clean, grade_enrollment_pivot = pivot_enrollment_dataset(grade_enrollment, 'GRADE', drop_columns=common_column_drop, index_columns=enrollment_index_col)
gender_enrollment_clean, gender_enrollment_pivot = pivot_enrollment_dataset(gender_enrollment, 'GENDER', drop_columns=common_column_drop, index_columns=enrollment_index_col)
ethnicity_enrollment_clean, ethnicity_enrollment_pivot = pivot_enrollment_dataset(ethnicity_enrollment, 'ETHNICITY', drop_columns=common_column_drop, index_columns=enrollment_index_col)

In [11]:
grade_enrollment.head()

,YEAR,REGION,COUNTY NAME,DISTRICT,DISTRICT NAME,CHARTER STATUS,GRADE,ENROLLMENT
0,2024-2025,10.0,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,Pre-kindergarten,147
1,2024-2025,10.0,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,Kindergarten,94
2,2024-2025,10.0,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,Grade 1,117
3,2024-2025,10.0,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,Grade 2,92
4,2024-2025,10.0,DALLAS COUNTY,57829.0,A+ ACADEMY,OPEN ENROLLMENT CHARTER,Grade 3,98


In [12]:
grade_enrollment_pivot.columns

Index(['County_Name', 'District_Name', 'Early Education', 'Grade 1',
       'Grade 10', 'Grade 11', 'Grade 12', 'Grade 2', 'Grade 3', 'Grade 4',
       'Grade 5', 'Grade 6', 'Grade 7', 'Grade 8', 'Grade 9', 'Kindergarten',
       'Pre-kindergarten'],
      dtype='str', name='GRADE')

In [13]:
econ_disadv_clean.replace(-999.0, np.nan, inplace=True)

,County_Name,District_Name,Eligible For Free Meals Count,Eligible For Free Meals Percent,Eligible For Reduced-Price Meals Count,Eligible For Reduced-Price Meals Percent,Other Economically Disadvantaged Count,Other Economically Disadvantaged Percent,Not Economically Disadvantaged Count,Not Economically Disadvantaged Percent,Total_Count
0,DALLAS COUNTY,A+ ACADEMY,1440.0,85.82,150.0,8.94,0.0,0.00,88.0,5.24,1678.0
1,HARRIS COUNTY,A+ UNLIMITED POTENTIAL,141.0,69.46,20.0,9.85,0.0,0.00,42.0,20.69,203.0
2,HILL COUNTY,ABBOTT ISD,87.0,30.63,22.0,7.75,0.0,0.00,175.0,61.62,284.0
3,HALE COUNTY,ABERNATHY ISD,297.0,36.76,NaN,NaN,NaN,NaN,415.0,51.36,808.0
4,TAYLOR COUNTY,ABILENE ISD,7205.0,49.57,134.0,0.92,2652.0,18.24,4545.0,31.27,14536.0
...,...,...,...,...,...,...,...,...,...,...,...
1203,DEWITT COUNTY,YORKTOWN ISD,61.0,10.63,16.0,2.79,NaN,NaN,279.0,48.61,574.0
1204,EL PASO COUNTY,YSLETA ISD,19286.0,56.62,1313.0,3.85,6308.0,18.52,7155.0,21.01,34062.0
1205,ZAPATA COUNTY,ZAPATA COUNTY ISD,2202.0,66.93,0.0,0.00,722.0,21.95,366.0,11.12,3290.0
1206,ANGELINA COUNTY,ZAVALLA ISD,220.0,72.61,0.0,0.00,0.0,0.00,83.0,27.39,303.0


In [14]:
pd.merge(staff_district_clean, econ_disadv_clean)

,County_Name,District_Name,FTE Count,Total Base Pay,Average Base Pay,Eligible For Free Meals Count,Eligible For Free Meals Percent,Eligible For Reduced-Price Meals Count,Eligible For Reduced-Price Meals Percent,Other Economically Disadvantaged Count,Other Economically Disadvantaged Percent,Not Economically Disadvantaged Count,Not Economically Disadvantaged Percent,Total_Count


In [15]:
#grade_enrollment_percent = convert_enrollment_percentage(grade_enrollment_pivot, econ_disadv_clean)
#gender_enrollment_percent = convert_enrollment_percentage(gender_enrollment_pivot, econ_disadv_clean)
#ethnicity_enrollment_percent = convert_enrollment_percentage(ethnicity_enrollment_pivot, econ_disadv_clean)
grade_enrollment_percent = replace_unknown_random_percent(grade_enrollment_pivot, econ_disadv_clean)
gender_enrollment_percent = replace_unknown_random_percent(gender_enrollment_pivot, econ_disadv_clean)
ethnicity_enrollment_percent = replace_unknown_random_percent(ethnicity_enrollment_pivot, econ_disadv_clean)

In [16]:
grade_enrollment_percent.head(10)

,County_Name,District_Name,Early Education,Grade 1,Grade 10,Grade 11,Grade 12,Grade 2,Grade 3,Grade 4,Grade 5,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten,Total_Count
0,ANDERSON COUNTY,CAYUGA ISD,0.012,0.087,0.092,0.088,0.064,0.069,0.075,0.073,0.069,0.071,0.066,0.078,0.085,0.071,0.012,577.0
1,ANDERSON COUNTY,ELKHART ISD,0.006,0.078,0.073,0.063,0.031,0.068,0.068,0.067,0.072,0.077,0.088,0.074,0.066,0.083,0.068,1187.0
2,ANDERSON COUNTY,FRANKSTON ISD,0.009,0.079,0.086,0.083,0.073,0.079,0.058,0.069,0.076,0.079,0.063,0.069,0.078,0.065,0.007,811.0
3,ANDERSON COUNTY,NECHES ISD,0.000,0.062,0.050,0.074,0.080,0.062,0.065,0.107,0.074,0.071,0.077,0.074,0.065,0.092,0.045,337.0
4,ANDERSON COUNTY,PALESTINE ISD,0.002,0.066,0.083,0.079,0.072,0.065,0.073,0.065,0.070,0.075,0.071,0.073,0.081,0.024,0.067,3186.0
5,ANDERSON COUNTY,SLOCUM ISD,0.022,0.077,0.056,0.059,0.071,0.105,0.062,0.071,0.065,0.084,0.074,0.074,0.080,0.065,0.028,323.0
6,ANDERSON COUNTY,WESTWOOD ISD,0.005,0.069,0.078,0.079,0.060,0.073,0.076,0.062,0.080,0.073,0.066,0.078,0.076,0.075,0.034,1387.0
7,ANDREWS COUNTY,ANDREWS ISD,0.005,0.072,0.076,0.071,0.069,0.077,0.078,0.081,0.071,0.079,0.077,0.074,0.077,0.068,0.026,4209.0
8,ANGELINA COUNTY,CENTRAL ISD,0.005,0.063,0.075,0.068,0.055,0.081,0.084,0.081,0.077,0.067,0.086,0.071,0.077,0.071,0.025,1497.0
9,ANGELINA COUNTY,DIBOLL ISD,0.004,0.076,0.072,0.083,0.080,0.061,0.068,0.030,0.067,0.084,0.070,0.071,0.074,0.063,0.067,1582.0


In [17]:
enrollment_percents = pd.merge(pd.merge(gender_enrollment_percent, ethnicity_enrollment_percent), grade_enrollment_percent)

In [18]:
#grade_enrollment_percent['County_Name'] =  grade_enrollment_percent['County_Name'].str.replace(' COUNTY', '')
#grade_enrollment_percent
enrollment_percents['County_Name'] =  enrollment_percents['County_Name'].str.replace(' COUNTY', '')
enrollment_percents

,County_Name,District_Name,Female,Male,Total_Count,American Indian or Alaska Nat,Asian,Black or African American,Hispanic/Latino,Native Hawaiian/Other Pacific,...,Grade 2,Grade 3,Grade 4,Grade 5,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten
0,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.000,0.007,0.062,0.104,0.000,...,0.069,0.075,0.073,0.069,0.071,0.066,0.078,0.085,0.071,0.012
1,ANDERSON,ELKHART ISD,0.465,0.535,1187.0,0.003,0.003,0.038,0.139,0.003,...,0.068,0.068,0.067,0.072,0.077,0.088,0.074,0.066,0.083,0.068
2,ANDERSON,FRANKSTON ISD,0.478,0.522,811.0,0.005,0.005,0.081,0.138,0.000,...,0.079,0.058,0.069,0.076,0.079,0.063,0.069,0.078,0.065,0.007
3,ANDERSON,NECHES ISD,0.504,0.496,337.0,0.012,0.012,0.071,0.154,0.000,...,0.062,0.065,0.107,0.074,0.071,0.077,0.074,0.065,0.092,0.045
4,ANDERSON,PALESTINE ISD,0.470,0.530,3186.0,0.001,0.010,0.236,0.436,0.000,...,0.065,0.073,0.065,0.070,0.075,0.071,0.073,0.081,0.024,0.067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1203,YOUNG,NEWCASTLE ISD,0.480,0.520,227.0,0.000,0.000,0.018,0.141,0.000,...,0.040,0.062,0.088,0.062,0.048,0.066,0.070,0.079,0.057,0.084
1204,YOUNG,OLNEY ISD,0.469,0.531,691.0,0.006,0.006,0.020,0.373,0.000,...,0.081,0.084,0.064,0.087,0.065,0.069,0.067,0.067,0.072,0.080
1205,ZAPATA,ZAPATA COUNTY ISD,0.479,0.521,3290.0,0.000,0.001,0.001,0.996,0.000,...,0.066,0.063,0.071,0.075,0.071,0.074,0.076,0.077,0.057,0.052
1206,ZAVALA,CRYSTAL CITY ISD,0.484,0.516,1643.0,0.000,0.000,0.002,0.985,0.000,...,0.029,0.061,0.075,0.077,0.077,0.072,0.066,0.077,0.062,0.075


In [19]:
staff_district_clean.head()

,County_Name,District_Name,FTE Count,Total Base Pay,Average Base Pay
0,DALLAS COUNTY,A+ ACADEMY,4.04,278424.0,68946
1,DALLAS COUNTY,A+ ACADEMY,8.18,577027.0,70541
2,DALLAS COUNTY,A+ ACADEMY,2.91,218878.0,75203
3,DALLAS COUNTY,A+ ACADEMY,5.64,421021.0,74686
4,DALLAS COUNTY,A+ ACADEMY,4.99,365122.0,73130


In [20]:
staff_district_clean['County_Name'] =  staff_district_clean['County_Name'].str.replace(' COUNTY', '')
enrollment_salary = pd.merge(enrollment_percents, staff_district_clean)
enrollment_salary.head()

,County_Name,District_Name,Female,Male,Total_Count,American Indian or Alaska Nat,Asian,Black or African American,Hispanic/Latino,Native Hawaiian/Other Pacific,...,Grade 5,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten,FTE Count,Total Base Pay,Average Base Pay
0,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.069,0.071,0.066,0.078,0.085,0.071,0.012,0.67,41984.0,63039
1,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.069,0.071,0.066,0.078,0.085,0.071,0.012,2.81,142000.0,50581
2,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.069,0.071,0.066,0.078,0.085,0.071,0.012,0.13,6224.0,48549
3,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.069,0.071,0.066,0.078,0.085,0.071,0.012,4.46,245433.0,55050
4,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.069,0.071,0.066,0.078,0.085,0.071,0.012,1.28,70560.0,55229


In [21]:
tx_mmr_vac_rates_prep_merge = tx_mmr_vac_rates.rename(columns={'County': 'County_Name', 'School District or Name': 'District_Name'})
tx_mmr_vac_rates_prep_merge['County_Name'] = tx_mmr_vac_rates_prep_merge['County_Name'].str.upper()
#tx_mmr_vac_rates_prep_merge.columns
tx_mmr_vac_rates_prep_merge[['District_Name', 'County_Name', 'MMR Vaccination Rate']]

,District_Name,County_Name,MMR Vaccination Rate
0,A+ ACADEMY,DALLAS,96.77
1,ABBOTT ISD,HILL,95.24
2,ABERNATHY ISD,HALE,93.22
3,ABILENE ISD,TAYLOR,97.76
4,ACADEMY ISD,BELL,98.04
...,...,...,...
3338,ZAPATA COUNTY ISD,ZAPATA,100.00
3339,ZAVALLA ISD,ANGELINA,100.00
3340,ZEPHYR ISD,BROWN,100.00
3341,Zion Lutheran School Dallas,DALLAS,100.00


In [ ]:
enrollment_percents_vac_rates = pd.merge(enrollment_percents, tx_mmr_vac_rates_prep_merge[['District_Name', 'County_Name', 'MMR Vaccination Rate']])
#enrollment_percents_vac_rates.to_csv("../feature_inputs/tea_districts_enroll_percent_regression.csv", index=False)
enrollment_percents_vac_rates.to_csv("../feature_inputs/tea_districts_enroll_percent_regression_rand.csv", index=False)
enrollment_percents_vac_rates.head()

,County_Name,District_Name,Female,Male,Total_Count,American Indian or Alaska Nat,Asian,Black or African American,Hispanic/Latino,Native Hawaiian/Other Pacific,...,Grade 3,Grade 4,Grade 5,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten,MMR Vaccination Rate
0,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.000,0.007,0.062,0.104,0.000,...,0.075,0.073,0.069,0.071,0.066,0.078,0.085,0.071,0.012,90.24
1,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.000,0.007,0.062,0.104,0.000,...,0.075,0.073,0.069,0.071,0.066,0.078,0.085,0.071,0.012,94.74
2,ANDERSON,ELKHART ISD,0.465,0.535,1187.0,0.003,0.003,0.038,0.139,0.003,...,0.068,0.067,0.072,0.077,0.088,0.074,0.066,0.083,0.068,93.00
3,ANDERSON,ELKHART ISD,0.465,0.535,1187.0,0.003,0.003,0.038,0.139,0.003,...,0.068,0.067,0.072,0.077,0.088,0.074,0.066,0.083,0.068,93.14
4,ANDERSON,FRANKSTON ISD,0.478,0.522,811.0,0.005,0.005,0.081,0.138,0.000,...,0.058,0.069,0.076,0.079,0.063,0.069,0.078,0.065,0.007,96.23


In [ ]:
enrollment_salary_vac_rates = pd.merge(enrollment_salary, tx_mmr_vac_rates_prep_merge[['District_Name', 'County_Name', 'MMR Vaccination Rate']])
enrollment_salary_vac_rates['Average Base Pay'] = enrollment_salary_vac_rates['Average Base Pay'].replace('.', 0).astype(float)
#enrollment_salary_vac_rates.to_csv("../feature_inputs/tea_districts_enroll_percent_staff_salary_regression.csv", index=False)
enrollment_salary_vac_rates.to_csv("../feature_inputs/tea_districts_enroll_percent_staff_salary_regression_rand.csv", index=False)
enrollment_salary_vac_rates.head()

,County_Name,District_Name,Female,Male,Total_Count,American Indian or Alaska Nat,Asian,Black or African American,Hispanic/Latino,Native Hawaiian/Other Pacific,...,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten,FTE Count,Total Base Pay,Average Base Pay,MMR Vaccination Rate
0,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.071,0.066,0.078,0.085,0.071,0.012,0.67,41984.0,63039.0,90.24
1,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.071,0.066,0.078,0.085,0.071,0.012,0.67,41984.0,63039.0,94.74
2,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.071,0.066,0.078,0.085,0.071,0.012,2.81,142000.0,50581.0,90.24
3,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.071,0.066,0.078,0.085,0.071,0.012,2.81,142000.0,50581.0,94.74
4,ANDERSON,CAYUGA ISD,0.468,0.532,577.0,0.0,0.007,0.062,0.104,0.0,...,0.071,0.066,0.078,0.085,0.071,0.012,0.13,6224.0,48549.0,90.24


In [41]:
enrollment_salary_vac_rates.isna().any()

County_Name                      False
District_Name                    False
Female                           False
Male                             False
Total_Count                      False
American Indian or Alaska Nat    False
Asian                            False
Black or African American        False
Hispanic/Latino                  False
Native Hawaiian/Other Pacific    False
Two or more races                False
White                            False
Early Education                  False
Grade 1                          False
Grade 10                         False
Grade 11                         False
Grade 12                         False
Grade 2                          False
Grade 3                          False
Grade 4                          False
Grade 5                          False
Grade 6                          False
Grade 7                          False
Grade 8                          False
Grade 9                          False
Kindergarten             

In [24]:
tea_drop_columns = ['County_Name', 'District_Name', 'Total_Count']